In [ ]:
import os
import json
import shutil
import glob
import numpy as np
import torch

from compcats.utils import model_utils
from compcats.localizer import localizer

import sys
sys.path.append(os.path.join(os.getcwd(), '../../../../comparing-category-selectivity'))
from preprint import paths
from preprint import main_analysis as ma

MODELS_DIR = '../../../../comparing-category-selectivity/preprint/models'

ORDERED_MODELS = {
    'vitb16siglip2res224': 'ViT-B/16 SigLIP2',
    'resnet50openaiclip': 'ResNet-50 CLIP',
    'convnexttinyimagenet1kv1': 'ConvNeXt-Tiny ImageNet-1K',
    'resnet101openaiclip': 'ResNet-101 CLIP',
    'vitb14dinov2reg': 'ViT-B/14 DINOv2-Registers',
    'resnet50simclr': 'ResNet-50 SimCLR',
    'resnet50toponettau30': 'ResNet-50 TopoNet',
    'cornets': 'CORNet-S',
    'resnet101imagenet1kv1': 'ResNet-101 ImageNet-1K',
    'vitb14dinov2': 'ViT-B/14 DINOv2',
    'resnet50cesslbarlowtop': 'ResNet-50 CE-SSL',
    'resnet18imagenet1kv1': 'ResNet-18 ImageNet-1K',
    'resnet50imagenet1kv2': 'ResNet-50 ImageNet-1K',
    'resnet50faceobfuscation': 'ResNet-50 Face-Obfuscated',
    'vitb16eva028bs8bb131k': 'ViT-B/16 EVA-02',
    'resnet50ecoset': 'ResNet-50 EcoSet',
    'vitl14kosmos2': 'ViT-L/14 KOSMOS-2',
    'vitb16synclr': 'ViT-B/16 SynCLR',
    'resnet18tdannsimclrlw1': 'ResNet-18 TDANN-SimCLR',
    'vgg16imagenet1kv1': 'VGG-16 ImageNet-1K',
    'vitb32openaiclip': 'ViT-B/32 CLIP',
    'vitb16openaiclip': 'ViT-B/16 CLIP',
    'vitl14openaiclip': 'ViT-L/14 CLIP',
    'resnet50robustl2eps3': 'ResNet-50 Robust',
    'vitb32dreamsimopenclip': 'ViT-B/32 DreamSim',
    'resnet50blurstrong': 'ResNet-50 Blur-Trained',
    'resnet50sin': 'ResNet-50 Shape-Trained',
    'bagnet33': 'BagNet-33 ImageNet-1K',
    'vitb32imagenet1kv1': 'ViT-B/32 ImageNet-1K',
    'alexnetimagenet1kv1': 'AlexNet ImageNet-1K',
    'alexnetbarlowtwins': 'AlexNet BarlowTwins',
    'vitb16imagenet21k': 'ViT-B/16 ImageNet-21K',
    'resnet18tdannsupervisedlw10': 'ResNet-18 TDANN-Supervised',
    'cornetz': 'CORNet-Z',
    'alexnetipclref01': 'AlexNet IPCL',
    'alexnetrandom1': 'AlexNet 1',
    'alexnetrandom2': 'AlexNet 2',
    'alexnetrandom3': 'AlexNet 3',
    'alexnetrandom4': 'AlexNet 4',
    'alexnetrandom5': 'AlexNet 5',
    'resnet18random1': 'ResNet-18 1',
    'resnet18random2': 'ResNet-18 2',
    'resnet18random3': 'ResNet-18 3',
    'resnet18random4': 'ResNet-18 4',
    'resnet18random5': 'ResNet-18 5',
    'resnet50random1': 'ResNet-50 1',
    'resnet50random2': 'ResNet-50 2',
    'resnet50random3': 'ResNet-50 3',
    'resnet50random4': 'ResNet-50 4',
    'resnet50random5': 'ResNet-50 5',
    'vitb32random1': 'ViT-B/32 1',
    'vitb32random2': 'ViT-B/32 2',
    'vitb32random3': 'ViT-B/32 3',
    'vitb32random4': 'ViT-B/32 4',
    'vitb32random5': 'ViT-B/32 5',
}
UNTRAINED_MODELS = [model_name for model_name in ORDERED_MODELS if 'random' in model_name]

### Generate constant for representative layers

In [ ]:
# Generate layers constant
frois = ['FFA', 'EBA', 'PPA']
froi_to_domain = {'FFA': 'face', 'EBA': 'body', 'PPA': 'scene'}

# Build nested dict: model -> domain -> layer
model_layers = {}
for froi in frois:
    domain = froi_to_domain[froi]
    layers = ma.get_froi_models_layers(froi, froi)
    for model_name, layer in layers.items():
        if model_name not in model_layers:
            model_layers[model_name] = {}
        model_layers[model_name][domain] = layer

# Print as JS const
print('const MODEL_LAYERS = {')
for model_name, domains in model_layers.items():
    domain_str = ', '.join(f"{d}: '{l}'" for d, l in domains.items())
    print(f"  {model_name}: {{ {domain_str} }},")
print('};')

const MODEL_LAYERS = {
  alexnetimagenet1kv1: { face: 'features.11', body: 'features.12', scene: 'features.12' },
  alexnetipclref01: { face: 'conv_block_5.1', body: 'conv_block_5.3', scene: 'conv_block_5.0' },
  alexnetbarlowtwins: { face: 'backbone.4.1', body: 'backbone.4.2', scene: 'backbone.4.3' },
  vgg16imagenet1kv1: { face: 'features.30', body: 'features.30', scene: 'features.30' },
  resnet18imagenet1kv1: { face: 'layer4.1.relu1', body: 'layer4.1.relu2', scene: 'layer3.1.bn1' },
  resnet18tdannsupervisedlw10: { face: 'layer4.1.bn1', body: 'layer4.1.bn1', scene: 'layer4.0.conv2' },
  resnet18tdannsimclrlw1: { face: 'layer4.1.conv1', body: 'layer4.1.bn1', scene: 'layer4.1.conv1' },
  resnet50imagenet1kv2: { face: 'layer4.0.relu3', body: 'layer3.3.bn1', scene: 'layer3.2.bn3' },
  resnet50openaiclip: { face: 'layer4.1.relu1', body: 'layer4.0.bn3', scene: 'layer4.2.relu3' },
  resnet50simclr: { face: 'avgpool', body: 'avgpool', scene: 'layer3.5.conv1' },
  resnet50robustl2eps3: { fa

### Generate independent stimuli responses

In [18]:
model_names = list(ORDERED_MODELS.keys())
frois = ['FFA', 'EBA', 'PPA']
froi_to_domain = {'FFA': 'face', 'EBA': 'body', 'PPA': 'scene'}
tthresholds = [3, 5, 7, 9, 11]
ordered_categories = ['faces', 'bodies', 'scenes', 'objects']

localizer_paths = {
    'grillspector': paths.GRILLSPECTOR_LOCALIZER_DIR,
    'kanwisher':    paths.KANWISHER_LOCALIZER_DIR,
    'nsd':          paths.NSD_DIR
}

# One output dict per localizer
output = {loc: {} for loc in localizer_paths}

for model_name in model_names:
    # model = ma.get_model(model_name)
    # model.eval()
    print(f'Processing {model_name}...')

    for froi in frois:
        domain = froi_to_domain[froi]
        layer  = ma.get_froi_models_layers(froi, froi)[model_name]

        for model_localizer, localizer_path in localizer_paths.items():
            if model_localizer == 'nsd':
                if model_name in UNTRAINED_MODELS:
                    tthresholds_list = [3]
                else:
                    tthresholds_list = [7]
            else:
                tthresholds_list = tthresholds

            for tthreshold in tthresholds_list:
                # acts = model_utils.get_dataset_activations(
                #     MODELS_DIR, paths.PRINCE2024_LOCALIZER_DIR,
                #     model, layer,
                #     dataset='localizer',
                #     localizer_type='prince2024',
                #     froi=froi, tthreshold=tthreshold,
                #     froi_localizer=model_localizer,
                #     froi_localizer_dir=localizer_path
                # )

                # Get total units in layer for % calculation
                suffix = ''
                if model_localizer == 'nsd':
                    suffix = '_0.00'
                froi_mask = torch.load(
                    f'{MODELS_DIR}/{model_name}/{model_localizer}/{froi}/{layer}/{model_name}_{layer}_{model_localizer}_{froi}_{tthreshold:.2f}{suffix}.pt'
                )
                if model_localizer == 'nsd':
                    froi_mask = froi_mask['all']['mask']
                n_units = torch.sum(froi_mask).item()
                n_total = froi_mask.numel()

                # Error handling, in case of no units localized
                if n_units == 0:
                    entry = {
                        'n_units': 0,
                        'n_total': int(n_total),
                        'means':   [0, 0, 0, 0],
                        'q1':      [0, 0, 0, 0],
                        'q3':      [0, 0, 0, 0],
                        'wlo':     [0, 0, 0, 0],
                        'whi':     [0, 0, 0, 0],
                        'scatter': {cat: [] for cat in ordered_categories},
                    }
                    loc_out = output[model_localizer]
                    if model_name not in loc_out:
                        loc_out[model_name] = {}
                    if domain not in loc_out[model_name]:
                        loc_out[model_name][domain] = {}
                    loc_out[model_name][domain][str(tthreshold)] = entry
                    continue

                # acts: dict of category -> tensor (80 stimuli x N_units)
                cat_means = {}
                # Mean across units per stimulus → shape (80,)
                for cat in ordered_categories:
                    cat_acts = torch.load(
                        f'{MODELS_DIR}/{model_name}/prince2024/activations/{layer}/{model_name}_{layer}_prince2024_{cat}.pt'
                    )
                    cat_acts = cat_acts[:, froi_mask]
                    cat_means[cat] = cat_acts.mean(dim=1)  # dim=1 = units, not dim=0

                # Z-score globally across all stimuli and categories
                all_vals = torch.cat(list(cat_means.values()))  # (4 * 80,)
                overall_mean = all_vals.mean().item()
                overall_std  = all_vals.std().item()

                def zsc(t):
                    return ((t - overall_mean) / overall_std).cpu().numpy()

                # Per-category stats
                cat_data = {}
                scatter_rows = []  # one row per unit: [f_resp, b_resp, s_resp, o_resp]

                for cat in ordered_categories:
                    z = zsc(cat_means[cat])  # (N_units,)
                    q1 = float(np.percentile(z, 25))
                    q3 = float(np.percentile(z, 75))
                    iqr = q3 - q1
                    lo  = float(z[z >= q1 - 1.5*iqr].min())
                    hi  = float(z[z <= q3 + 1.5*iqr].max())
                    cat_data[cat] = {
                        'mean': float(z.mean()),
                        'q1':   q1,
                        'q3':   q3,
                        'wlo':  lo,
                        'whi':  hi,
                    }

                # shape: (80,) per category — 80 stimulus responses for that category
                scatter = {cat: zsc(cat_means[cat]).tolist() for cat in ordered_categories}

                entry = {
                    'n_units': int(n_units),
                    'n_total': int(n_total),
                    'means':   [cat_data[c]['mean'] for c in ordered_categories],
                    'q1':      [cat_data[c]['q1']   for c in ordered_categories],
                    'q3':      [cat_data[c]['q3']   for c in ordered_categories],
                    'wlo':     [cat_data[c]['wlo']  for c in ordered_categories],
                    'whi':     [cat_data[c]['whi']  for c in ordered_categories],
                    'scatter': {cat: zsc(cat_means[cat]).tolist() for cat in ordered_categories},
                }

                # Nest into output[localizer][model][domain][threshold]
                loc_out = output[model_localizer]
                if model_name not in loc_out:
                    loc_out[model_name] = {}
                if domain not in loc_out[model_name]:
                    loc_out[model_name][domain] = {}
                loc_out[model_name][domain][str(tthreshold)] = entry

# Save one file per localizer
import os
os.makedirs('data', exist_ok=True)
for model_localizer, data in output.items():
    path = f'data/fig1demo_{model_localizer}.json'
    with open(path, 'w') as f:
        json.dump(data, f)
    print(f'Saved {path}')

Processing vitb16siglip2res224...
Processing resnet50openaiclip...
Processing convnexttinyimagenet1kv1...
Processing resnet101openaiclip...
Processing vitb14dinov2reg...
Processing resnet50simclr...
Processing resnet50toponettau30...
Processing cornets...
Processing resnet101imagenet1kv1...
Processing vitb14dinov2...
Processing resnet50cesslbarlowtop...
Processing resnet18imagenet1kv1...
Processing resnet50imagenet1kv2...
Processing resnet50faceobfuscation...
Processing vitb16eva028bs8bb131k...
Processing resnet50ecoset...
Processing vitl14kosmos2...
Processing vitb16synclr...
Processing resnet18tdannsimclrlw1...
Processing vgg16imagenet1kv1...
Processing vitb32openaiclip...
Processing vitb16openaiclip...
Processing vitl14openaiclip...
Processing resnet50robustl2eps3...
Processing vitb32dreamsimopenclip...
Processing resnet50blurstrong...
Processing resnet50sin...
Processing bagnet33...
Processing vitb32imagenet1kv1...
Processing alexnetimagenet1kv1...
Processing alexnetbarlowtwins...


### Generate untrained model masks

In [ ]:
# Generate all untrained model masks
for model_name in UNTRAINED_MODELS:
    model = ma.get_model(model_name)
    # model.eval()
    print(f'Processing {model_name}...')

    for froi in frois:
        layer  = ma.get_froi_models_layers(froi, froi)[model_name]
        for model_localizer, localizer_path in localizer_paths.items():
            if model_localizer == 'nsd':
                continue
            for tthreshold in tthresholds:
                froi_mask = localizer.get_froi_mask(
                    localizer_path, model_localizer, MODELS_DIR,
                    model, layer, froi, tthreshold
                )

Initialized alexnetrandom1 with seed = 1
Processing alexnetrandom1...
Saving fROI ../../../../comparing-category-selectivity/preprint/models/alexnetrandom1/kanwisher/FFA/features.12/alexnetrandom1_features.12_kanwisher_FFA_5.00.pt
Saving fROI ../../../../comparing-category-selectivity/preprint/models/alexnetrandom1/kanwisher/FFA/features.12/alexnetrandom1_features.12_kanwisher_FFA_7.00.pt
Saving fROI ../../../../comparing-category-selectivity/preprint/models/alexnetrandom1/kanwisher/FFA/features.12/alexnetrandom1_features.12_kanwisher_FFA_9.00.pt
Saving fROI ../../../../comparing-category-selectivity/preprint/models/alexnetrandom1/kanwisher/FFA/features.12/alexnetrandom1_features.12_kanwisher_FFA_11.00.pt
Saving fROI ../../../../comparing-category-selectivity/preprint/models/alexnetrandom1/kanwisher/EBA/features.12/alexnetrandom1_features.12_kanwisher_EBA_5.00.pt
Saving fROI ../../../../comparing-category-selectivity/preprint/models/alexnetrandom1/kanwisher/EBA/features.12/alexnetrando

### Store independent localizer stimuli

In [21]:
cat_map = {
    'faces':   '1-Faces',
    'bodies':  '2-Bodies',
    'scenes':  '3-Scenes',
    'objects': '5-Objects',
}

for cat, folder in cat_map.items():
    src_files = sorted(glob.glob(f'{paths.PRINCE2024_LOCALIZER_DIR}/{folder}/*.jpg'))
    dst_dir = f'../../images/independent_stimuli/{cat}'
    os.makedirs(dst_dir, exist_ok=True)
    for idx, src in enumerate(src_files):
        shutil.copy(src, f'{dst_dir}/{idx:03d}.jpg')
    print(f'{cat}: {len(src_files)} images')

faces: 80 images
bodies: 80 images
scenes: 80 images
objects: 80 images


### Conditions where no units were identified

In [1]:
import json

for localizer in ['grillspector', 'kanwisher', 'nsd']:
    with open(f'../../data/fig1demo_{localizer}.json') as f:
        data = json.load(f)
    
    print(f'\n=== {localizer} ===')
    for model in data:
        if model == 'image_files':
            continue
        for domain in data[model]:
            for thresh, entry in data[model][domain].items():
                if entry['n_units'] == 0:
                    print(f'  {model} | {domain} | t={thresh} → 0 units')


=== grillspector ===
  resnet50sin | scene | t=11 → 0 units
  alexnetrandom1 | body | t=9 → 0 units
  alexnetrandom1 | body | t=11 → 0 units
  alexnetrandom1 | scene | t=5 → 0 units
  alexnetrandom1 | scene | t=7 → 0 units
  alexnetrandom1 | scene | t=9 → 0 units
  alexnetrandom1 | scene | t=11 → 0 units
  alexnetrandom2 | body | t=11 → 0 units
  alexnetrandom2 | scene | t=5 → 0 units
  alexnetrandom2 | scene | t=7 → 0 units
  alexnetrandom2 | scene | t=9 → 0 units
  alexnetrandom2 | scene | t=11 → 0 units
  alexnetrandom3 | body | t=9 → 0 units
  alexnetrandom3 | body | t=11 → 0 units
  alexnetrandom3 | scene | t=5 → 0 units
  alexnetrandom3 | scene | t=7 → 0 units
  alexnetrandom3 | scene | t=9 → 0 units
  alexnetrandom3 | scene | t=11 → 0 units
  alexnetrandom4 | body | t=9 → 0 units
  alexnetrandom4 | body | t=11 → 0 units
  alexnetrandom4 | scene | t=5 → 0 units
  alexnetrandom4 | scene | t=7 → 0 units
  alexnetrandom4 | scene | t=9 → 0 units
  alexnetrandom4 | scene | t=11 → 0 u